# Final 02.2: Mixed Emotion Phase 2 with Final Model-Specific Prompt Policy

Runs Llama 2 v2 CoT and the established Llama 3 SELF-DISCOVER protocol on the Phase 1-routed Mixed Emotion examples, then exports model-specific Phase 2 outputs for end-to-end orchestration.

Final prompt policy:
- Llama 2: universal-policy-v2 CoT.
- Llama 3: established baseline SELF-DISCOVER protocol.

This notebook is the frozen model-specific Phase 2 configuration. It does not overwrite outputs from prior prompt-policy experiments.


In [ ]:
# Colab setup. Run this cell first in a fresh Colab GPU runtime.
# IMPORTANT: after this cell finishes, restart the runtime once, then run from the imports cell.
%pip install -q -U pandas tqdm scikit-learn sentencepiece protobuf accelerate transformers "bitsandbytes>=0.46.1"

import importlib.metadata as importlib_metadata
print("bitsandbytes:", importlib_metadata.version("bitsandbytes"))
print("transformers:", importlib_metadata.version("transformers"))
print("accelerate:", importlib_metadata.version("accelerate"))

print("\nSETUP COMPLETE. Now restart the runtime once: Runtime > Restart runtime, then rerun from the import cell.")


In [ ]:
import os
import gc
import re
import json
import hashlib
import shutil
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from tqdm.auto import tqdm

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

try:
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
except Exception:
    accuracy_score = classification_report = confusion_matrix = None

LABELS = ["Depression", "Neutral", "Happy"]




In [ ]:
# Dependency check. Run this after restarting the runtime.
import importlib.metadata as importlib_metadata
import importlib.util

bnb_spec = importlib.util.find_spec("bitsandbytes")
print("bitsandbytes spec:", bnb_spec)
print("bitsandbytes version:", importlib_metadata.version("bitsandbytes"))

if bnb_spec is None:
    raise RuntimeError("bitsandbytes is not importable. Re-run the setup cell, restart runtime, and rerun from the imports cell.")


## Configuration

For final paper results, this notebook should read the Phase 1 output produced by Final 01:

`/content/drive/MyDrive/confidence_guided_llm_reasoning/outputs_final/phase1_distilbert/phase1_mixed_emotion_predictions.csv`

The notebook then filters to `phase1_routed=True`. Only those low-confidence Phase 1 examples are sent to Llama 2 / Llama 3. Accepted Phase 1 examples are not reprocessed here; they are handled later by Final 03 during orchestration.

All outputs are saved to Google Drive under `outputs_final/phase2_llm_reasoning_model_specific_final/`, and each completed row is appended immediately so the run can resume after a Colab disconnect.


## Universal Prompt Policy v2 Experiment

This experimental version strengthens the prompt policy for mixed or shifting emotion cases. It instructs both Llama 2 CoT and Llama 3 SELF-DISCOVER to avoid defaulting to Neutral merely because multiple cues are present, and to use the final emotional trajectory and overall takeaway when the post moves from one emotional state to another.


In [ ]:
# Mixed Emotion v2.4 fallback source. Final 02 normally reads the Phase 1 output from Final 01.
MIXED_EMOTION_DATA_URL = (
    "https://raw.githubusercontent.com/WoojinPark-Jay/"
    "confidence-guided-llm-reasoning-depression-risk-emotion/"
    "a90393a07fbc468959f165b3ae9b04ddf669c94b/"
    "data/supplementary/mixed_emotion/"
    "mixed_emotion_stress_test_v2_4_neutral_clear_300.csv"
)

# Final Phase 1 output. This is produced by 01_distilbert_phase1_training_final_colab.ipynb.
PHASE1_PREDICTIONS_PATH = Path("/content/drive/MyDrive/confidence_guided_llm_reasoning/outputs_final/phase1_distilbert/phase1_mixed_emotion_predictions.csv")

# Save outputs to Google Drive so row-level checkpoints survive runtime resets.
USE_GOOGLE_DRIVE_OUTPUT = True
REQUIRE_PERSISTENT_OUTPUT = True
LOCAL_OUTPUT_DIR = Path("outputs_final/phase2_llm_reasoning_model_specific_final")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/confidence_guided_llm_reasoning/outputs_final/phase2_llm_reasoning_model_specific_final")

OUTPUT_DIR = LOCAL_OUTPUT_DIR
DRIVE_OUTPUT_AVAILABLE = False

if USE_GOOGLE_DRIVE_OUTPUT:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        probe_path = DRIVE_OUTPUT_DIR / "_drive_write_test.txt"
        probe_path.write_text("ok", encoding="utf-8")
        probe_path.unlink(missing_ok=True)
        OUTPUT_DIR = DRIVE_OUTPUT_DIR
        DRIVE_OUTPUT_AVAILABLE = True
        print(f"Google Drive output enabled: {OUTPUT_DIR}")
    except Exception as exc:
        if REQUIRE_PERSISTENT_OUTPUT:
            raise RuntimeError(
                "Google Drive output is unavailable, so the notebook stopped before running expensive inference. "
                "Fix Drive authorization/mount first, or set REQUIRE_PERSISTENT_OUTPUT = False only for a temporary smoke test."
            ) from exc
        print(f"Google Drive output is unavailable ({exc}); falling back to local runtime output.")
        OUTPUT_DIR = LOCAL_OUTPUT_DIR
else:
    if REQUIRE_PERSISTENT_OUTPUT:
        raise RuntimeError("Turn on Drive output or set REQUIRE_PERSISTENT_OUTPUT=False for temporary smoke tests.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEXT_COL = "text"
TRUE_LABEL_COL = "target_label"
PHASE1_LABEL_COL = "phase1_label"
PHASE1_ROUTED_COL = "phase1_routed"

# Final end-to-end mode: use only rows routed by Phase 1.
RUN_ROUTED_ONLY = True
# If Phase 1 file is missing, allow all Mixed Emotion rows for prompt validation.
ALLOW_ALL_ROWS_WITHOUT_PHASE1 = True
MAX_ROWS = None  # final run uses all routed rows; set small integer for smoke testing.
RANDOM_STATE = 42

RUN_LLAMA2_COT = True
RUN_LLAMA3_SELF_DISCOVER = True

LLAMA2_MODEL_NAME = "NousResearch/Llama-2-7b-chat-hf"
LLAMA3_MODEL_NAME = "NousResearch/Meta-Llama-3-8B-Instruct"
SELF_DISCOVER_STRUCTURE_MODE = "per_sample"

MAX_NEW_TOKENS_COT = 256
MAX_NEW_TOKENS_SELF_DISCOVER = 1024

LLAMA2_OUTPUT_PATH = OUTPUT_DIR / "llama2_cot_routed_mixed_emotion_results.csv"
LLAMA3_OUTPUT_PATH = OUTPUT_DIR / "llama3_self_discover_routed_mixed_emotion_results.csv"
RESUME_FROM_EXISTING = True
print(f"Output directory: {OUTPUT_DIR}")


In [ ]:
# Optional Hugging Face token support for gated or rate-limited model access.
# In Colab, add a secret named HF_TOKEN if needed.
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
        print("HF_TOKEN loaded from Colab secrets.")
except Exception:
    print("Colab userdata not available or HF_TOKEN not set. Continuing without explicit HF token.")


## Load Mixed Emotion Dataset

In [ ]:
def load_phase2_input():
    if PHASE1_PREDICTIONS_PATH.exists():
        print("Loading Phase 1 predictions:", PHASE1_PREDICTIONS_PATH)
        df = pd.read_csv(PHASE1_PREDICTIONS_PATH)
        required = {"example_id", TEXT_COL, "target_label", PHASE1_LABEL_COL}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"Phase 1 prediction file missing columns: {missing}")
        df["phase1_label_for_prompt"] = df[PHASE1_LABEL_COL]
        df["input_source"] = "phase1_predictions"
        return df

    if not ALLOW_ALL_ROWS_WITHOUT_PHASE1:
        raise FileNotFoundError(f"Phase 1 prediction file not found: {PHASE1_PREDICTIONS_PATH}")

    print("Phase 1 prediction file not found. Loading all Mixed Emotion examples for prompt validation:")
    print(MIXED_EMOTION_DATA_URL)
    df = pd.read_csv(MIXED_EMOTION_DATA_URL)
    df["phase1_label_for_prompt"] = df[TRUE_LABEL_COL]
    df["phase1_routed"] = True
    df["phase1_label"] = df[TRUE_LABEL_COL]
    df["input_source"] = "mixed_emotion_all_rows_without_phase1"
    return df

df = load_phase2_input()
print(df.shape)
display(df.head())
print(df[TRUE_LABEL_COL].value_counts())


In [ ]:
required = {"example_id", TEXT_COL, TRUE_LABEL_COL, "phase1_label_for_prompt"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

work_df = df.copy()
if RUN_ROUTED_ONLY and PHASE1_ROUTED_COL in work_df.columns:
    work_df = work_df[work_df[PHASE1_ROUTED_COL].astype(str).str.lower().isin(["true", "1", "yes"])]
    print(f"Using routed rows only: {len(work_df)} rows")
elif RUN_ROUTED_ONLY:
    print("RUN_ROUTED_ONLY=True but phase1_routed column is absent. Running all loaded rows.")

if MAX_ROWS is not None:
    work_df = work_df.sample(n=min(MAX_ROWS, len(work_df)), random_state=RANDOM_STATE).reset_index(drop=True)
else:
    work_df = work_df.reset_index(drop=True)

if work_df.empty:
    raise ValueError("No rows selected for Phase 2 reasoning. Check threshold/routing settings.")

print(work_df.shape)
display(work_df[["example_id", TRUE_LABEL_COL, "phase1_label_for_prompt", TEXT_COL]].head())
# The selected input CSV is written after the output-safety guard below.


In [ ]:
## Phase 2 output safety guard
# This cell prevents old v2.3/v2.4 Phase 2 CSV files from being mixed with the
# current Phase 1 routed set. If the Phase 1 fingerprint changed, old CSV files
# are moved to a timestamped backup folder automatically.

def _phase2_fingerprint(source_df, selected_df):
    source_cols = [
        col for col in [
            "example_id",
            TRUE_LABEL_COL,
            PHASE1_LABEL_COL,
            PHASE1_ROUTED_COL,
            "phase1_confidence",
            "routing_threshold",
            "temperature",
            "prompt_version",
        ]
        if col in source_df.columns
    ]
    source_part = source_df[source_cols].copy().sort_values("example_id").reset_index(drop=True)

    selected_cols = [col for col in ["example_id", TRUE_LABEL_COL, "phase1_label_for_prompt"] if col in selected_df.columns]
    selected_part = selected_df[selected_cols].copy().sort_values("example_id").reset_index(drop=True)

    payload = {
        "source_rows": int(len(source_df)),
        "selected_rows": int(len(selected_df)),
        "source_csv": source_part.to_csv(index=False),
        "selected_csv": selected_part.to_csv(index=False),
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()


def _phase2_manifest_payload(source_df, selected_df):
    threshold = None
    if "routing_threshold" in source_df.columns and source_df["routing_threshold"].notna().any():
        threshold = float(source_df["routing_threshold"].dropna().iloc[0])

    temperature = None
    if "temperature" in source_df.columns and source_df["temperature"].notna().any():
        temperature = float(source_df["temperature"].dropna().iloc[0])

    prompt_version = None
    if "prompt_version" in source_df.columns and source_df["prompt_version"].notna().any():
        prompt_version = str(source_df["prompt_version"].dropna().iloc[0])

    fingerprint = _phase2_fingerprint(source_df, selected_df)
    run_id = f"mixed_routed{len(selected_df)}"
    if threshold is not None:
        run_id += f"_tau{threshold:.2f}".replace(".", "p")
    run_id += f"_{fingerprint[:8]}"

    return {
        "run_id": run_id,
        "phase1_fingerprint": fingerprint,
        "created_at_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
        "phase1_predictions_path": str(PHASE1_PREDICTIONS_PATH),
        "phase1_rows": int(len(source_df)),
        "selected_routed_rows": int(len(selected_df)),
        "run_routed_only": bool(RUN_ROUTED_ONLY),
        "routing_threshold": threshold,
        "temperature": temperature,
        "prompt_version": prompt_version,
        "llama2_output_path": str(LLAMA2_OUTPUT_PATH),
        "llama3_output_path": str(LLAMA3_OUTPUT_PATH),
    }


def prepare_phase2_outputs(source_df, selected_df):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    manifest_path = OUTPUT_DIR / "phase2_run_manifest.json"
    current_manifest = _phase2_manifest_payload(source_df, selected_df)

    previous_manifest = None
    if manifest_path.exists():
        try:
            previous_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        except Exception:
            previous_manifest = None

    same_phase1_input = (
        previous_manifest is not None
        and previous_manifest.get("phase1_fingerprint") == current_manifest["phase1_fingerprint"]
    )

    if same_phase1_input:
        print("Phase 2 output guard: same Phase 1 routed set detected. Resuming existing CSV outputs.")
        print("Run ID:", previous_manifest.get("run_id"))
    else:
        existing_csvs = sorted(OUTPUT_DIR.glob("*.csv"))
        if existing_csvs:
            backup_dir = OUTPUT_DIR / f"backup_before_{current_manifest['run_id']}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
            backup_dir.mkdir(parents=True, exist_ok=True)
            for csv_path in existing_csvs:
                shutil.move(str(csv_path), str(backup_dir / csv_path.name))
            print("Phase 2 output guard: previous CSV outputs were backed up automatically.")
            print("Backup folder:", backup_dir)
        else:
            print("Phase 2 output guard: no previous CSV outputs found. Starting clean.")

    manifest_path.write_text(json.dumps(current_manifest, indent=2), encoding="utf-8")
    work_df.to_csv(OUTPUT_DIR / "phase2_selected_input_rows.csv", index=False)
    print("Current Phase 2 run ID:", current_manifest["run_id"])
    print("Selected routed rows:", len(selected_df), "/", len(source_df))
    print("Manifest:", manifest_path)
    print("Selected input CSV:", OUTPUT_DIR / "phase2_selected_input_rows.csv")


prepare_phase2_outputs(df, work_df)


## Model Loading Helpers

In [ ]:
def load_chat_model(model_name, load_in_4bit=True):
    compute_dtype = torch.float16
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
        token=os.environ.get("HF_TOKEN"),
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    quantization_config = None
    if load_in_4bit:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=True,
        )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=compute_dtype,
        device_map="auto",
        quantization_config=quantization_config,
        trust_remote_code=True,
        token=os.environ.get("HF_TOKEN"),
    )
    model.eval()
    return tokenizer, model

def clear_model(tokenizer=None, model=None):
    del tokenizer
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## Appendix B-Aligned Llama 2 Chain-of-Thought Prompt

In [ ]:
PROMPT_POLICY_VERSION = "final-model-specific-policy-v1"

LLAMA2_COT_REQUESTS = [
    "You are an expert annotator for research-oriented, non-clinical emotion classification. Assist in analyzing emotions in text data. Do not make a clinical diagnosis, infer a medical condition, or provide treatment advice.",
    """I will first provide a piece of text. Independently assess its emotional content before comparing it with a Phase 1 AI-generated label. The only permitted labels are Depression, Neutral, and Happy.""",
    """Independently analyze the text using the classification policy below. State the dominant emotion and the textual evidence before considering the Phase 1 label.

Classification Policy:
- Depression: unresolved sadness, hopelessness, emotional distress, emotional exhaustion, withdrawal, self-devaluation, or a clearly negative overall trajectory is dominant.
- Neutral: the text is mainly factual, routine, balanced, informational, or emotionally mild, without a dominant positive or distress-related state.
- Happy: happiness, relief, gratitude, accomplishment, fulfillment, or a clearly positive resolution is dominant.

For every text, assess the dominant emotional meaning of the full text. Do not decide from isolated words, brief cues, or a simple average of positive and negative expressions. When the text contains multiple emotional cues or a clear temporal emotional shift, additionally consider the overall trajectory and final takeaway. Do not assume a trajectory when the text does not clearly support one. Do not select Neutral merely because multiple cues are present. A brief positive cue does not make the text Happy when unresolved distress remains dominant, and a brief negative cue does not make the text Depression when the text clearly resolves into sustained relief or positive resolution.

Use only Depression, Neutral, or Happy for your provisional label.""",
    """The Phase 1 classifier predicted: {phase1_label}

Compare that prediction with your independent assessment. Confirm it only when it is supported by the dominant emotional meaning of the full text. Otherwise, explain the correction using textual evidence only.""",
    """Provide the final Phase 2 decision. Use only one exact label: Depression, Neutral, or Happy. Do not use synonyms or additional labels such as Sad, Positive, Mixed, Anxiety, or Other. Do not provide a percentage breakdown.

End the response with exactly one line: Final label: [label]""",
]


def format_llama2_text_input(text):
    return f"Text:\n{text}"


In [ ]:
def run_llama2_cot_one(tokenizer, model, text, phase1_label):
    messages = [
        {"role": "system", "content": LLAMA2_COT_REQUESTS[0]},
        {"role": "user", "content": LLAMA2_COT_REQUESTS[1]},
    ]
    messages.append({"role": "assistant", "content": chat_generate(tokenizer, model, messages, MAX_NEW_TOKENS_COT, do_sample=True, temperature=0.6, top_p=0.9)})
    messages.append({"role": "user", "content": format_llama2_text_input(text)})
    messages.append({"role": "assistant", "content": chat_generate(tokenizer, model, messages, MAX_NEW_TOKENS_COT, do_sample=True, temperature=0.6, top_p=0.9)})

    independent_prompt = LLAMA2_COT_REQUESTS[2]
    messages.append({"role": "user", "content": independent_prompt})
    independent_answer = chat_generate(tokenizer, model, messages, MAX_NEW_TOKENS_COT, do_sample=True, temperature=0.6, top_p=0.9)
    messages.append({"role": "assistant", "content": independent_answer})

    comparison_prompt = LLAMA2_COT_REQUESTS[3].format(phase1_label=phase1_label)
    messages.append({"role": "user", "content": comparison_prompt})
    comparison_answer = chat_generate(tokenizer, model, messages, MAX_NEW_TOKENS_COT, do_sample=True, temperature=0.6, top_p=0.9)
    messages.append({"role": "assistant", "content": comparison_answer})

    messages.append({"role": "user", "content": LLAMA2_COT_REQUESTS[4]})
    final_answer = chat_generate(tokenizer, model, messages, MAX_NEW_TOKENS_COT, do_sample=True, temperature=0.6, top_p=0.9)
    return [independent_answer, comparison_answer, final_answer]


def parse_llama2_final_label(output):
    text = str(output)
    match = re.search(r"Final label\s*:\s*(Depression|Neutral|Happy)", text, flags=re.IGNORECASE)
    if match:
        return match.group(1).capitalize()

    fallback_patterns = [
        r"final (?:phase 2 )?(?:classification )?label[^.\n:]*[:\s]+(Depression|Neutral|Happy)",
        r"correct classification[^.\n:]*[:\s]+(Depression|Neutral|Happy)",
        r"dominant emotion[^.\n:]*[:\s]+(Depression|Neutral|Happy)",
        r"classif(?:y|ied|ication)[^.\n]*\b(Depression|Neutral|Happy)\b",
    ]
    for pattern in fallback_patterns:
        fallback = re.search(pattern, text, flags=re.IGNORECASE)
        if fallback:
            return fallback.group(1).capitalize()
    return np.nan


In [ ]:
def _normalize_id(value):
    if pd.isna(value):
        return None
    return str(value)


def load_existing_results(path):
    path = Path(path)
    if RESUME_FROM_EXISTING and path.exists() and path.stat().st_size > 0:
        existing = pd.read_csv(path)
        if "example_id" in existing.columns:
            existing = existing.drop_duplicates(subset=["example_id"], keep="last")
        return existing
    return pd.DataFrame()


def completed_example_ids(path):
    existing = load_existing_results(path)
    if existing.empty or "example_id" not in existing.columns:
        return set()
    return set(existing["example_id"].map(_normalize_id).dropna())


def append_result_row(path, row_dict):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([row_dict]).to_csv(
        path,
        mode="a",
        header=not path.exists(),
        index=False,
    )


In [ ]:
# Shared chat-generation helper. This must run before Llama 2 and Llama 3 execution cells.
def format_messages_without_chat_template(messages):
    """Fallback formatter for chat models whose tokenizer has no chat_template."""
    system_parts = [m["content"] for m in messages if m.get("role") == "system"]
    dialogue = [m for m in messages if m.get("role") != "system"]
    system_text = "\n".join(system_parts).strip()

    prompt = ""
    pending_user = None
    first_turn = True
    for message in dialogue:
        role = message.get("role")
        content = str(message.get("content", "")).strip()
        if role == "user":
            pending_user = content
        elif role == "assistant" and pending_user is not None:
            if first_turn and system_text:
                prompt += f"<s>[INST] <<SYS>>\n{system_text}\n<</SYS>>\n\n{pending_user} [/INST] {content} </s>"
            else:
                prompt += f"<s>[INST] {pending_user} [/INST] {content} </s>"
            pending_user = None
            first_turn = False

    if pending_user is not None:
        if first_turn and system_text:
            prompt += f"<s>[INST] <<SYS>>\n{system_text}\n<</SYS>>\n\n{pending_user} [/INST]"
        else:
            prompt += f"<s>[INST] {pending_user} [/INST]"
    return prompt

def build_chat_inputs(tokenizer, model, messages):
    if getattr(tokenizer, "chat_template", None):
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        prompt = format_messages_without_chat_template(messages)

    encoded = tokenizer(prompt, return_tensors="pt")
    return {key: value.to(model.device) for key, value in encoded.items()}

def chat_generate(tokenizer, model, messages, max_new_tokens=256, do_sample=False, temperature=None, top_p=None):
    model_inputs = build_chat_inputs(tokenizer, model, messages)
    input_ids = model_inputs["input_ids"]

    terminators = [tokenizer.eos_token_id]
    eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    if isinstance(eot_id, int) and eot_id >= 0:
        terminators.append(eot_id)

    kwargs = dict(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        eos_token_id=terminators,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=do_sample,
    )
    if temperature is not None:
        kwargs["temperature"] = temperature
    if top_p is not None:
        kwargs["top_p"] = top_p

    with torch.no_grad():
        outputs = model.generate(**kwargs)
    response = outputs[0][input_ids.shape[-1]:]
    return tokenizer.decode(response, skip_special_tokens=True).strip()


In [ ]:
llama2_results = load_existing_results(LLAMA2_OUTPUT_PATH)

if RUN_LLAMA2_COT:
    completed_ids = completed_example_ids(LLAMA2_OUTPUT_PATH)
    pending_df = work_df[~work_df["example_id"].map(_normalize_id).isin(completed_ids)].copy()
    print(f"Llama 2 CoT resume: {len(completed_ids)} completed, {len(pending_df)} pending.")

    if pending_df.empty:
        print(f"No pending Llama 2 rows. Loaded existing results from: {LLAMA2_OUTPUT_PATH}")
    else:
        tokenizer, model = load_chat_model(LLAMA2_MODEL_NAME, load_in_4bit=True)
        for _, row in tqdm(pending_df.iterrows(), total=len(pending_df), desc="Llama 2 CoT"):
            answers = run_llama2_cot_one(
                tokenizer,
                model,
                row[TEXT_COL],
                row["phase1_label_for_prompt"],
            )
            result_row = {
                "example_id": row.get("example_id", None),
                "text": row[TEXT_COL],
                "target_label": row[TRUE_LABEL_COL],
                "phase1_label_for_prompt": row["phase1_label_for_prompt"],
                "prompt_policy_version": PROMPT_POLICY_VERSION,
                "LLaMA2_1": answers[0] if len(answers) > 0 else np.nan,
                "LLaMA2_2": answers[1] if len(answers) > 1 else np.nan,
                "LLaMA2_3": answers[2] if len(answers) > 2 else np.nan,
            }
            result_row["LLaMA2_final_label"] = parse_llama2_final_label(result_row["LLaMA2_3"])
            append_result_row(LLAMA2_OUTPUT_PATH, result_row)
        clear_model(tokenizer, model)

    llama2_results = load_existing_results(LLAMA2_OUTPUT_PATH)
    print(f"Saved/resumed: {LLAMA2_OUTPUT_PATH}")
    print(f"Llama 2 rows available: {len(llama2_results)}")
    display(llama2_results.head())


## Appendix C-Aligned Llama 3 SELF-DISCOVER Prompt

The SELF-DISCOVER select/adapt/implement prompt templates are embedded here from `prompts.py`, so this notebook does not need a separate `prompts.py` upload in Colab.


In [ ]:
reasoning_modules = """
1 How could I devise an experiment to help solve that problem?
2 Make a list of ideas for solving this problem, and apply them one by one to the problem to see if any progress can be made.
3 How could I measure progress on this problem?
4 How can I simplify the problem so that it is easier to solve?
5 What are the key assumptions underlying this problem?
6 What are the potential risks and drawbacks of each solution?
7 What are the alternative perspectives or viewpoints on this problem?
8 What are the long-term implications of this problem and its solutions?
9 How can I break down this problem into smaller, more manageable parts?
10 Critical Thinking: This style involves analyzing the problem from different perspectives, questioning assumptions, and evaluating
the evidence or information available. It focuses on logical reasoning, evidence-based decision-making, and identifying
potential biases or flaws in thinking.
11 Try creative thinking, generate innovative and out-of-the-box ideas to solve the problem. Explore unconventional solutions,
thinking beyond traditional boundaries, and encouraging imagination and originality.
12 Seek input and collaboration from others to solve the problem. Emphasize teamwork, open communication, and leveraging the
diverse perspectives and expertise of a group to come up with effective solutions.
13 Use systems thinking: Consider the problem as part of a larger system and understanding the interconnectedness of various elements.
Focuses on identifying the underlying causes, feedback loops, and interdependencies that influence the problem, and developing holistic
solutions that address the system as a whole.
14 Use Risk Analysis: Evaluate potential risks, uncertainties, and tradeoffs associated with different solutions or approaches to a
problem. Emphasize assessing the potential consequences and likelihood of success or failure, and making informed decisions based
on a balanced analysis of risks and benefits.
15 Use Reflective Thinking: Step back from the problem, take the time for introspection and self-reflection. Examine personal biases,
assumptions, and mental models that may influence problem-solving, and being open to learning from past experiences to improve
future approaches.
16 What is the core issue or problem that needs to be addressed?
17 What are the underlying causes or factors contributing to the problem?
18 Are there any potential solutions or strategies that have been tried before? If yes, what were the outcomes and lessons learned?
19 What are the potential obstacles or challenges that might arise in solving this problem?
20 Are there any relevant data or information that can provide insights into the problem? If yes, what data sources are available,
and how can they be analyzed?
21 Are there any stakeholders or individuals who are directly affected by the problem? What are their perspectives and needs?
22 What resources (financial, human, technological, etc.) are needed to tackle the problem effectively?
23 How can progress or success in solving the problem be measured or evaluated?
24 What indicators or metrics can be used?
25 Is the problem a technical or practical one that requires a specific expertise or skill set? Or is it more of a conceptual or
theoretical problem?
26 Does the problem involve a physical constraint, such as limited resources, infrastructure, or space?
27 Is the problem related to human behavior, such as a social, cultural, or psychological issue?
28 Does the problem involve decision-making or planning, where choices need to be made under uncertainty or with competing
objectives?
29 Is the problem an analytical one that requires data analysis, modeling, or optimization techniques?
30 Is the problem a design challenge that requires creative solutions and innovation?
31 Does the problem require addressing systemic or structural issues rather than just individual instances?
32 Is the problem time-sensitive or urgent, requiring immediate attention and action?
33 What kinds of solution typically are produced for this kind of problem specification?
34 Given the problem specification and the current best solution, have a guess about other possible solutions.
35 Let's imagine the current best solution is totally wrong, what other ways are there to think about the problem specification?
36 What is the best way to modify this current best solution, given what you know about these kinds of problem specification?
37 Ignoring the current best solution, create an entirely new solution to the problem.
38 Let's think step by step.
39 Let's make a step by step plan and implement it with good notion and explanation"""


select_prompt = """
In order to solve the given task:
<Task>
{Task}
</Task>
Select several modules that are crucial for solving the tasks above
from all the reasoning module description given below:
{resonining_modules}
"""

adapt_prompt = """
Rephrase and specify each reasoning module so that it better helps solving the task:
<Task>
{Task}
</Task>
SELECTED module descriptions:
{selected_modules}
Adapt each reasoning module description to better solve the task:
"""

implement_prompt = """
Operationalize the reasoning modules into a step-by-step reasoning plan in JSON format
Example task:
<Task>
{Task}
</Task>
ADAPTED module descriptions:
{adapted_modules}

Implement a reasoning structure to generalise similar task to follow step-by-step and arrive at correct answers
"""

In [ ]:
SELF_DISCOVER_TASK_TEMPLATE = """
You are an expert annotator for mental-health-related emotion classification.
Your task is to analyze the emotional content of text data using structured reasoning.
This task is intended for research-oriented text classification, not clinical diagnosis.

I will provide you with text data and an AI-generated emotional classification label.
Your task is to determine the dominant emotion that best represents the overall sentiment of the text.
The text may contain multiple emotions, but your goal is to determine the most representative emotion that captures the overall tone and final emotional trajectory.
Do not make a clinical diagnosis, infer a medical condition, or provide treatment advice.

<context>

data: {data}

result: {label}

<questions>

1. Objectively analyze the given text. Identify the dominant emotion by considering the text's overall sentiment, final emotional trajectory, and final takeaway.
2. Compare your analysis from Question 1 with the AI's classified label.
3. Evaluate the AI's classification using textual evidence. If it aligns with your independent assessment, confirm it. If it does not, determine the correct dominant emotion and justify the decision using only evidence from the text.
4. Select the option that most accurately represents your analysis. Do not create or explain additional choices beyond the provided options. End the response with exactly one final label using the format Final label: [label].
<Options>Depression, Neutral, Happy</Options>

Classification Guidelines:
- If the text expresses ongoing sadness, hopelessness, emotional distress, emotional exhaustion, self-devaluation, or a strongly negative emotional trajectory, classify it as Depression.
- If the text is mainly factual, balanced, informational, or emotionally mild, and does not contain a clear Depression- or Happy-oriented trajectory, classify it as Neutral.
- If the text expresses happiness, accomplishment, relief, gratitude, fulfillment, or positive resolution as the dominant sentiment, classify it as Happy.

Mixed and Shifting Emotion Handling:
For blended or emotionally shifting texts, first identify the final emotional trajectory and final takeaway of the post, then classify based on that final takeaway rather than averaging isolated emotional cues.
Do not classify a text as Neutral merely because it contains mixed cues.
If the text moves from distress toward relief, accomplishment, or positive resolution, classify it as Happy.
If the text moves from neutral or positive content toward hopelessness, emotional exhaustion, or unresolved distress, classify it as Depression.
Use Neutral only when the final takeaway remains primarily factual, balanced, or emotionally mild without a clear Depression- or Happy-oriented trajectory.

Output Constraint:
Do not create labels outside the provided options.
When explaining the decision, base the justification on textual evidence rather than clinical assumptions.
The response should end with exactly one final label in the format Final label: Depression, Final label: Neutral, or Final label: Happy.

</questions>
"""

FIXED_SELF_DISCOVER_STRUCTURE = """
Use a structured, text-grounded reasoning plan:
1. Identify the dominant emotional cues in the text.
2. Determine whether the text has a blended or shifting emotional trajectory.
3. Prioritize the final emotional trajectory and overall takeaway rather than averaging isolated cues.
4. Compare the resulting emotional classification with the AI-generated label.
5. Justify the decision using textual evidence only.
6. End with exactly one label in the format Final label: Depression, Final label: Neutral, or Final label: Happy.
"""

def build_self_discover_task(text, phase1_label):
    return SELF_DISCOVER_TASK_TEMPLATE.replace("{data}", str(text)).replace("{label}", str(phase1_label))

def parse_llama3_final_label(output):
    text = str(output)
    match = re.search(r"Final label\s*:\s*(Depression|Neutral|Happy)", text, flags=re.IGNORECASE)
    if match:
        value = match.group(1).lower()
        return next(label for label in LABELS if label.lower() == value)
    for label in LABELS:
        if re.search(rf"\b{label}\b", text, flags=re.IGNORECASE):
            return label
    return np.nan



In [ ]:
# Llama 3 can be run after a partial Colab execution. If the shared chat helper
# cell above was skipped, define the same helper functions here as a fallback.
if "chat_generate" not in globals():
    def format_messages_without_chat_template(messages):
        """Fallback formatter for chat models whose tokenizer has no chat_template."""
        system_parts = [m["content"] for m in messages if m.get("role") == "system"]
        dialogue = [m for m in messages if m.get("role") != "system"]
        system_text = "\n".join(system_parts).strip()

        prompt = ""
        pending_user = None
        first_turn = True
        for message in dialogue:
            role = message.get("role")
            content = str(message.get("content", "")).strip()
            if role == "user":
                pending_user = content
            elif role == "assistant" and pending_user is not None:
                if first_turn and system_text:
                    prompt += f"<s>[INST] <<SYS>>\n{system_text}\n<</SYS>>\n\n{pending_user} [/INST] {content} </s>"
                else:
                    prompt += f"<s>[INST] {pending_user} [/INST] {content} </s>"
                pending_user = None
                first_turn = False

        if pending_user is not None:
            if first_turn and system_text:
                prompt += f"<s>[INST] <<SYS>>\n{system_text}\n<</SYS>>\n\n{pending_user} [/INST]"
            else:
                prompt += f"<s>[INST] {pending_user} [/INST]"
        return prompt

    def build_chat_inputs(tokenizer, model, messages):
        if getattr(tokenizer, "chat_template", None):
            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        else:
            prompt = format_messages_without_chat_template(messages)

        encoded = tokenizer(prompt, return_tensors="pt")
        return {key: value.to(model.device) for key, value in encoded.items()}

    def chat_generate(tokenizer, model, messages, max_new_tokens=256, do_sample=False, temperature=None, top_p=None):
        model_inputs = build_chat_inputs(tokenizer, model, messages)
        input_ids = model_inputs["input_ids"]

        terminators = [tokenizer.eos_token_id]
        eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
        if isinstance(eot_id, int) and eot_id >= 0:
            terminators.append(eot_id)

        kwargs = dict(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            eos_token_id=terminators,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=do_sample,
        )
        if temperature is not None:
            kwargs["temperature"] = temperature
        if top_p is not None:
            kwargs["top_p"] = top_p

        with torch.no_grad():
            outputs = model.generate(**kwargs)
        response = outputs[0][input_ids.shape[-1]:]
        return tokenizer.decode(response, skip_special_tokens=True).strip()

def run_self_discover_structure(tokenizer, model, task):
    select = select_prompt.replace("{Task}", task).replace("{resonining_modules}", reasoning_modules)
    selected_modules = chat_generate(tokenizer, model, [{"role": "user", "content": select}], MAX_NEW_TOKENS_SELF_DISCOVER, do_sample=True, top_p=1.0)

    adapt = adapt_prompt.replace("{Task}", task).replace("{selected_modules}", selected_modules)
    adapted_modules = chat_generate(tokenizer, model, [{"role": "user", "content": adapt}], MAX_NEW_TOKENS_SELF_DISCOVER, do_sample=True, top_p=1.0)

    implement = implement_prompt.replace("{Task}", task).replace("{adapted_modules}", adapted_modules)
    reasoning_structure = chat_generate(tokenizer, model, [{"role": "user", "content": implement}], MAX_NEW_TOKENS_SELF_DISCOVER, do_sample=True, top_p=1.0)
    return selected_modules, adapted_modules, reasoning_structure

def run_llama3_self_discover_one(tokenizer, model, text, phase1_label):
    task = build_self_discover_task(text, phase1_label)
    if SELF_DISCOVER_STRUCTURE_MODE == "per_sample":
        selected, adapted, structure = run_self_discover_structure(tokenizer, model, task)
    elif SELF_DISCOVER_STRUCTURE_MODE == "fixed":
        selected = "Fixed paper-safe reasoning modules: textual evidence review, label comparison, mixed/shifting emotion handling, final trajectory assessment, and final constrained label selection."
        adapted = "Adapted to identify dominant emotional cues, prioritize final emotional trajectory for blended texts, compare with the AI-generated label, and avoid clinical inference."
        structure = FIXED_SELF_DISCOVER_STRUCTURE
    else:
        raise ValueError("SELF_DISCOVER_STRUCTURE_MODE must be 'per_sample' or 'fixed'.")

    final_prompt = (
        f"Using the following reasoning structure:\n{structure}\n\n"
        f"Solve this task, providing your answer:\n{task}\n\n"
        "Note1: Write the question number before your answer.\n"
        "Note2: Do not write anything besides your answer.\n"
        "Note3: End with exactly one final label in the format Final label: Depression, Final label: Neutral, or Final label: Happy."
    )
    messages = [
        {"role": "system", "content": "You are an expert annotator for research-oriented, non-clinical emotion classification. Do not provide clinical diagnosis, medical inference, treatment advice, or professional mental health advice."},
        {"role": "user", "content": final_prompt},
    ]
    answer = chat_generate(tokenizer, model, messages, MAX_NEW_TOKENS_SELF_DISCOVER, do_sample=False)
    return selected, adapted, structure, answer


In [ ]:
llama3_results = load_existing_results(LLAMA3_OUTPUT_PATH)

if RUN_LLAMA3_SELF_DISCOVER:
    completed_ids = completed_example_ids(LLAMA3_OUTPUT_PATH)
    pending_df = work_df[~work_df["example_id"].map(_normalize_id).isin(completed_ids)].copy()
    print(f"Llama 3 SELF-DISCOVER resume: {len(completed_ids)} completed, {len(pending_df)} pending.")

    if pending_df.empty:
        print(f"No pending Llama 3 rows. Loaded existing results from: {LLAMA3_OUTPUT_PATH}")
    else:
        tokenizer, model = load_chat_model(LLAMA3_MODEL_NAME, load_in_4bit=True)
        for _, row in tqdm(pending_df.iterrows(), total=len(pending_df), desc="Llama 3 SELF-DISCOVER"):
            selected, adapted, structure, answer = run_llama3_self_discover_one(
                tokenizer,
                model,
                row[TEXT_COL],
                row["phase1_label_for_prompt"],
            )
            result_row = {
                "example_id": row.get("example_id", None),
                "text": row[TEXT_COL],
                "target_label": row[TRUE_LABEL_COL],
                "phase1_label_for_prompt": row["phase1_label_for_prompt"],
                "prompt_policy_version": PROMPT_POLICY_VERSION,
                "LLaMA3_SELECT": selected,
                "LLaMA3_ADAPT": adapted,
                "LLaMA3_IMPLEMENT": structure,
                "LLaMA3_Answer": answer,
            }
            result_row["LLaMA3_final_label"] = parse_llama3_final_label(result_row["LLaMA3_Answer"])
            append_result_row(LLAMA3_OUTPUT_PATH, result_row)
        clear_model(tokenizer, model)

    llama3_results = load_existing_results(LLAMA3_OUTPUT_PATH)
    print(f"Saved/resumed: {LLAMA3_OUTPUT_PATH}")
    print(f"Llama 3 rows available: {len(llama3_results)}")
    display(llama3_results.head())



## Evaluation

This section evaluates Phase 2 final labels against `target_label`. When Phase 1 predictions are added later, this can be expanded to correction counts, introduced errors, and net corrections.


In [ ]:
def _safe_classification_report(eval_df, pred_col, model_name):
    if classification_report is None or eval_df.empty:
        return pd.DataFrame()
    report = classification_report(
        eval_df["target_label"],
        eval_df[pred_col],
        labels=LABELS,
        output_dict=True,
        zero_division=0,
    )
    rows = []
    for label, values in report.items():
        if isinstance(values, dict):
            row = {"model": model_name, "label": label, **values}
        else:
            row = {"model": model_name, "label": label, "score": values}
        rows.append(row)
    return pd.DataFrame(rows)


def _save_phase2_confusion_matrix(eval_df, pred_col, model_name, filename_prefix):
    if confusion_matrix is None or eval_df.empty:
        return None, pd.DataFrame()
    cm = confusion_matrix(eval_df["target_label"], eval_df[pred_col], labels=LABELS)
    cm_df = pd.DataFrame(cm, index=LABELS, columns=LABELS)
    cm_csv_path = OUTPUT_DIR / f"{filename_prefix}_confusion_matrix.csv"
    cm_df.to_csv(cm_csv_path)

    plt.figure(figsize=(5.2, 4.2))
    sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Target")
    plt.title(f"{model_name} Phase 2 Confusion Matrix")
    plt.tight_layout()
    png_path = OUTPUT_DIR / f"{filename_prefix}_confusion_matrix.png"
    plt.savefig(png_path, dpi=220)
    plt.show()
    return png_path, cm_df


def evaluate_final_labels(result_df, pred_col, name, filename_prefix):
    if result_df is None or pred_col not in result_df.columns:
        print(f"{name}: no results to evaluate.")
        return None, pd.DataFrame(), pd.DataFrame()
    eval_df = result_df.dropna(subset=[pred_col]).copy()
    if eval_df.empty:
        print(f"{name}: no parseable final labels.")
        return None, pd.DataFrame(), pd.DataFrame()

    acc = (eval_df[pred_col] == eval_df["target_label"]).mean()
    label_counts = eval_df[pred_col].value_counts().reindex(LABELS, fill_value=0).rename_axis("predicted_label").reset_index(name="count")
    label_counts.insert(0, "model", name)
    label_counts.to_csv(OUTPUT_DIR / f"{filename_prefix}_predicted_label_counts.csv", index=False)

    print(f"\n{name}")
    print(f"Rows evaluated: {len(eval_df)} / {len(result_df)}")
    print(f"Accuracy vs target_label: {acc:.4f}")
    cm_path, cm_df = _save_phase2_confusion_matrix(eval_df, pred_col, name, filename_prefix)
    display(cm_df)
    display(label_counts)

    report_df = _safe_classification_report(eval_df, pred_col, name)
    if not report_df.empty:
        report_path = OUTPUT_DIR / f"{filename_prefix}_classification_report.csv"
        report_df.to_csv(report_path, index=False)
        display(report_df)

    parse_failures = result_df[result_df[pred_col].isna()].copy()
    if not parse_failures.empty:
        parse_failures.to_csv(OUTPUT_DIR / f"{filename_prefix}_parse_failures.csv", index=False)

    return {
        "model": name,
        "rows_total": len(result_df),
        "rows_evaluated": len(eval_df),
        "parse_failures": int(result_df[pred_col].isna().sum()),
        "accuracy": float(acc),
        "confusion_matrix_png": str(cm_path) if cm_path else None,
    }, report_df, label_counts

summary = []
all_reports = []
all_label_counts = []
item, report, counts = evaluate_final_labels(llama2_results, "LLaMA2_final_label", "Llama 2 CoT", "llama2_cot")
if item:
    summary.append(item)
if not report.empty:
    all_reports.append(report)
if not counts.empty:
    all_label_counts.append(counts)

item, report, counts = evaluate_final_labels(llama3_results, "LLaMA3_final_label", "Llama 3 SELF-DISCOVER", "llama3_self_discover")
if item:
    summary.append(item)
if not report.empty:
    all_reports.append(report)
if not counts.empty:
    all_label_counts.append(counts)

summary_df = pd.DataFrame(summary)
if not summary_df.empty:
    summary_path = OUTPUT_DIR / "phase2_llm_reasoning_model_specific_final_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"Saved: {summary_path}")
    display(summary_df)

if all_reports:
    phase2_reports_df = pd.concat(all_reports, ignore_index=True)
    phase2_reports_df.to_csv(OUTPUT_DIR / "phase2_classification_reports.csv", index=False)
if all_label_counts:
    phase2_label_counts_df = pd.concat(all_label_counts, ignore_index=True)
    phase2_label_counts_df.to_csv(OUTPUT_DIR / "phase2_predicted_label_counts.csv", index=False)



## Combined Output Table

In [ ]:
combined = work_df[["example_id", TEXT_COL, TRUE_LABEL_COL, "phase1_label_for_prompt"]].copy()
if llama2_results is not None:
    combined = combined.merge(
        llama2_results[["example_id", "LLaMA2_1", "LLaMA2_2", "LLaMA2_3", "LLaMA2_final_label"]],
        on="example_id",
        how="left",
    )
if llama3_results is not None:
    combined = combined.merge(
        llama3_results[["example_id", "LLaMA3_SELECT", "LLaMA3_ADAPT", "LLaMA3_IMPLEMENT", "LLaMA3_Answer", "LLaMA3_final_label"]],
        on="example_id",
        how="left",
    )
combined_path = OUTPUT_DIR / "phase2_llm_reasoning_model_specific_final_combined_outputs.csv"
combined.to_csv(combined_path, index=False)
print(f"Saved: {combined_path}")
display(combined.head())


## Final export and local download

This cell packages all available result CSV files and starts a browser download.


In [ ]:
# Final export / download cell
# Run this after Llama 2 and/or Llama 3 cells finish.
# If OUTPUT_DIR is Google Drive, files are already persistent. This cell also creates one zip for easy local download.

from pathlib import Path
import zipfile

existing_output_files = []
for pattern in ["*.csv", "*.json", "*.png", "*.xlsx"]:
    existing_output_files.extend(sorted(OUTPUT_DIR.rglob(pattern)))
existing_output_files = sorted(set(Path(p) for p in existing_output_files))

print("Existing output files:")
for p in existing_output_files:
    print(f"- {p} | {p.stat().st_size:,} bytes")

if not existing_output_files:
    print("No output files found yet. Run the Llama 2/Llama 3 reasoning cells first.")
else:
    zip_path = OUTPUT_DIR / "phase2_llm_reasoning_model_specific_final_outputs.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in existing_output_files:
            if p == zip_path:
                continue
            zf.write(p, arcname=p.relative_to(OUTPUT_DIR) if p.is_relative_to(OUTPUT_DIR) else p.name)
    print(f"Saved zip: {zip_path} | {zip_path.stat().st_size:,} bytes")

    if DRIVE_OUTPUT_AVAILABLE:
        print("Persistent Drive copy is available here:")
        print(zip_path)
    else:
        print("WARNING: Drive output is not available. Download the zip before the Colab runtime disconnects.")

    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception as exc:
        print(f"Automatic browser download was not started: {exc}")
        print("If needed, use the Colab file browser or run files.download(str(zip_path)) manually.")

